In [15]:
## Cell 1 — Imports & Paths

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    confusion_matrix, classification_report,
)
import joblib

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────
DATA_PATH   = "D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/data/processed/model_dataset.csv"
MODELS_DIR  = "D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models"
MODEL_NAME  = "CreditRiskEngine_GBM_v2"

os.makedirs(MODELS_DIR, exist_ok=True)

# ── Reproducibility ───────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.20

print("=" * 60)
print("  CREDIT RISK ENGINE — MODEL TRAINING")
print(f"  Model  : {MODEL_NAME}")
print(f"  Run    : {datetime.now().strftime('%Y-%m-%dT%H:%M:%S')}")
print("=" * 60)

  CREDIT RISK ENGINE — MODEL TRAINING
  Model  : CreditRiskEngine_GBM_v2
  Run    : 2026-03-29T13:42:06


In [17]:
# 86 features produced by the Feature Engineering Pipeline
FEATURE_COLS = [
    # ── Behavioral (DPD / MoB) ───────────────────────────────
    "max_dpd_ever", "max_overdue_amount", "total_overdue_amount",
    "total_bounces", "total_mob", "max_mob", "bounce_rate",
    "max_dpd_12m_mob", "max_dpd_6m_mob", "last_delinq_mob",
    "ever_npa_flag", "months_since_last_delinquency",
    "chronic_delinquent", "recent_stress_flag",
    # ── Loan terms ───────────────────────────────────────────
    "sanction_amount", "interest_rate", "tenure_months",
    # ── Bureau ───────────────────────────────────────────────
    "bureau_score", "bureau_score_band_num",
    "total_tradelines", "secured_tradelines", "unsecured_tradelines",
    "credit_card_tradelines", "total_outstanding", "total_sanctioned_limit",
    "credit_utilisation_pct", "recent_enquiries_3m", "recent_enquiries_6m",
    "max_dpd_12m", "max_dpd_24m", "number_of_defaults",
    "oldest_account_vintage_months", "written_off_accounts",
    "suit_filed_flag", "wilful_default_flag",
    "ever_dpd90_12m", "ever_dpd90_24m", "has_writeoff",
    "high_enquiry_flag", "long_vintage_flag",
    "unsecured_mix_ratio", "enquiry_per_tradeline",
    # ── Bank statement ────────────────────────────────────────
    "avg_monthly_credit", "avg_monthly_debit", "avg_eod_balance",
    "min_balance", "max_balance",
    "emi_bounce_count", "salary_credit_count", "salary_flag",
    "inward_cheque_bounce_count", "outward_cheque_bounce_count",
    "avg_monthly_emi_outflow", "cash_withdrawal_pct",
    "upi_txn_count_monthly",
    "international_txn_flag", "gaming_txn_flag", "casino_txn_flag",
    "net_cashflow_ratio", "emi_to_income_ratio", "balance_to_income_ratio",
    "high_stress_flag", "high_cash_flag",
    # ── Application / income ─────────────────────────────────
    "declared_income", "verified_income", "itr_assessed_income",
    "work_experience_years", "years_at_current_employer",
    "requested_amount", "requested_tenure",
    "foir", "existing_monthly_obligations",
    "co_applicant_flag", "co_applicant_income",
    "ltv_ratio", "property_value",
    "income_deviation_pct", "mismatch_flag",
    "employment_risk_rank",
    "high_foir_flag", "high_ltv_flag", "low_experience",
    "declared_to_verified_ratio", "itr_to_declared_ratio",
    "effective_income", "loan_to_income_ratio",
]

TARGET_COL = "target_default"

print(f"  Total features defined : {len(FEATURE_COLS)}")

  Total features defined : 86


In [19]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("=" * 60)
print("  STEP 3 — LOAD MASTER DATASET")
print(f"  Shape          : {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"  Default rate   : {df[TARGET_COL].mean()*100:.2f}%")
print("=" * 60)

# ── Drop features not present in this dataset ─────────────────
available = [c for c in FEATURE_COLS if c in df.columns]
missing   = [c for c in FEATURE_COLS if c not in df.columns]

if missing:
    print(f"\n  ⚠  {len(missing)} feature(s) not found in dataset — dropping:")
    for f in missing:
        print(f"     • {f}")
else:
    print(f"\n  All {len(FEATURE_COLS)} features present ✓")

FEATURE_COLS = available     # use only what exists
print(f"\n  Features used  : {len(FEATURE_COLS)}")

# ── Null check on feature columns ─────────────────────────────
null_pct = df[FEATURE_COLS].isnull().mean() * 100
high_null = null_pct[null_pct > 5]
if not high_null.empty:
    print("\n  ⚠  Columns with >5% nulls (will be median-imputed):")
    print(high_null.sort_values(ascending=False).to_string())
    for col in high_null.index:
        df[col] = df[col].fillna(df[col].median())
else:
    print("  Null check     : clean ✓")

print(f"\n  Target distribution:")
print(df[TARGET_COL].value_counts().rename(index={0:"Good (0)", 1:"Bad (1)"}).to_string())

  STEP 3 — LOAD MASTER DATASET
  Shape          : 300 rows × 111 cols
  Default rate   : 51.67%

  All 86 features present ✓

  Features used  : 86
  Null check     : clean ✓

  Target distribution:
target_default
Bad (1)     155
Good (0)    145


In [21]:
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    stratify     = y,
    random_state = RANDOM_STATE,
)

print("=" * 60)
print("  STEP 4 — TRAIN / TEST SPLIT")
print(f"  Train  : {X_train.shape[0]:,} rows  |  Default rate : {y_train.mean()*100:.2f}%")
print(f"  Test   : {X_test.shape[0]:,} rows   |  Default rate : {y_test.mean()*100:.2f}%")
print("=" * 60)

  STEP 4 — TRAIN / TEST SPLIT
  Train  : 240 rows  |  Default rate : 51.67%
  Test   : 60 rows   |  Default rate : 51.67%


In [23]:
print("=" * 60)
print("  STEP 5 — GRADIENT BOOSTING MACHINE")
print("=" * 60)

GBM_PARAMS = {
    "n_estimators":       200,
    "learning_rate":      0.05,
    "max_depth":          4,
    "min_samples_leaf":   10,
    "subsample":          0.8,
    "max_features":       "sqrt",
    "random_state":       RANDOM_STATE,
}

gbm = GradientBoostingClassifier(**GBM_PARAMS)
gbm.fit(X_train, y_train)

# ── Cross-validation (stratified 5-fold) ─────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(gbm, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)

print(f"\n  CV AUC scores  : {[round(s,4) for s in cv_scores]}")
print(f"  CV mean AUC    : {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}")

# ── Test-set metrics ──────────────────────────────────────────
gbm_proba  = gbm.predict_proba(X_test)[:, 1]
gbm_pred   = (gbm_proba >= 0.5).astype(int)
gbm_auc    = roc_auc_score(y_test, gbm_proba)
gbm_gini   = 2 * gbm_auc - 1

print(f"\n  Test AUC       : {gbm_auc:.4f}")
print(f"  Test Gini      : {gbm_gini:.4f}")
print(f"  CV mean AUC    : {cv_scores.mean():.4f}")
print(f"\n  Confusion Matrix (threshold = 0.50):")
cm = confusion_matrix(y_test, gbm_pred)
print(f"  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")
print(f"\n{classification_report(y_test, gbm_pred, target_names=['Good','Bad'])}")
print("=" * 60)

  STEP 5 — GRADIENT BOOSTING MACHINE

  CV AUC scores  : [0.673, 0.7565, 0.7322, 0.9148, 0.8333]
  CV mean AUC    : 0.7820  ±  0.0840

  Test AUC       : 0.7330
  Test Gini      : 0.4661
  CV mean AUC    : 0.7820

  Confusion Matrix (threshold = 0.50):
  TN=17  FP=12  FN=10  TP=21

              precision    recall  f1-score   support

        Good       0.63      0.59      0.61        29
         Bad       0.64      0.68      0.66        31

    accuracy                           0.63        60
   macro avg       0.63      0.63      0.63        60
weighted avg       0.63      0.63      0.63        60



In [25]:
print("=" * 60)
print("  STEP 6 — LOGISTIC REGRESSION")
print("=" * 60)

# Scale features — LR is sensitive to magnitude
scaler    = StandardScaler()
X_tr_sc   = scaler.fit_transform(X_train)
X_te_sc   = scaler.transform(X_test)

lr = LogisticRegression(
    C            = 0.1,
    max_iter     = 1000,
    solver       = "lbfgs",
    random_state = RANDOM_STATE,
)
lr.fit(X_tr_sc, y_train)

lr_proba  = lr.predict_proba(X_te_sc)[:, 1]
lr_pred   = (lr_proba >= 0.5).astype(int)
lr_auc    = roc_auc_score(y_test, lr_proba)
lr_gini   = 2 * lr_auc - 1

print(f"\n  Test AUC       : {lr_auc:.4f}")
print(f"  Test Gini      : {lr_gini:.4f}")
print(f"\n  Confusion Matrix (threshold = 0.50):")
cm_lr = confusion_matrix(y_test, lr_pred)
print(f"  TN={cm_lr[0,0]}  FP={cm_lr[0,1]}  FN={cm_lr[1,0]}  TP={cm_lr[1,1]}")
print(f"\n  GBM vs LR  →  AUC delta : {gbm_auc - lr_auc:+.4f}")
print("=" * 60)

  STEP 6 — LOGISTIC REGRESSION

  Test AUC       : 0.6830
  Test Gini      : 0.3660

  Confusion Matrix (threshold = 0.50):
  TN=17  FP=12  FN=9  TP=22

  GBM vs LR  →  AUC delta : +0.0501


In [27]:
def ks_statistic(y_true, y_prob):
    """Return KS statistic and the threshold at which it occurs."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    ks  = (tpr - fpr).max()
    thr = thresholds[(tpr - fpr).argmax()]
    return round(float(ks), 4), round(float(thr), 4)

gbm_ks, gbm_ks_thr = ks_statistic(y_test, gbm_proba)
lr_ks,  lr_ks_thr  = ks_statistic(y_test, lr_proba)

print("=" * 60)
print("  STEP 7 — KS STATISTIC")
print(f"  GBM  →  KS = {gbm_ks:.4f}  at threshold {gbm_ks_thr:.4f}")
print(f"  LR   →  KS = {lr_ks:.4f}   at threshold {lr_ks_thr:.4f}")

# Status
def ks_status(ks):
    if ks >= 0.60: return "EXCELLENT"
    if ks >= 0.40: return "GOOD"
    if ks >= 0.20: return "ACCEPTABLE"
    return "WEAK"

def auc_status(auc):
    if auc >= 0.80: return "PASS"
    if auc >= 0.70: return "PASS"
    return "REVIEW"

print(f"\n  GBM  KS status  : {ks_status(gbm_ks)}")
print(f"  GBM  AUC status : {auc_status(gbm_auc)}")
print("=" * 60)

  STEP 7 — KS STATISTIC
  GBM  →  KS = 0.4127  at threshold 0.7408
  LR   →  KS = 0.3393   at threshold 0.6181

  GBM  KS status  : GOOD
  GBM  AUC status : PASS


In [29]:
def build_lift_table(y_true, y_prob, n_deciles=10):
    """Build decile-level lift table sorted by descending score."""
    result = pd.DataFrame({"y_true": y_true, "y_prob": y_prob})
    result = result.sort_values("y_prob", ascending=False).reset_index(drop=True)
    result["decile"] = pd.qcut(result.index, q=n_deciles, labels=False) + 1

    total_bads  = y_true.sum()
    total_goods = (1 - y_true).sum()
    overall_bad_rate = y_true.mean()

    rows = []
    for d in range(1, n_deciles + 1):
        seg       = result[result["decile"] == d]
        n         = len(seg)
        bads      = int(seg["y_true"].sum())
        goods     = n - bads
        bad_rate  = round(bads / n * 100, 2) if n else 0
        lift      = round(bad_rate / (overall_bad_rate * 100), 2) if overall_bad_rate else 0
        rows.append({
            "decile":      d,
            "n":           n,
            "bads":        bads,
            "goods":       goods,
            "bad_rate_pct": bad_rate,
            "lift":        lift,
        })

    lift_df = pd.DataFrame(rows)

    # Cumulative KS
    cum_bad  = lift_df["bads"].cumsum()  / total_bads  * 100
    cum_good = lift_df["goods"].cumsum() / total_goods * 100
    lift_df["cum_bad_pct"]  = cum_bad.round(2)
    lift_df["cum_good_pct"] = cum_good.round(2)
    lift_df["ks_pct"]       = (cum_bad - cum_good).round(2)

    return lift_df

gbm_lift_df = build_lift_table(y_test, gbm_proba)

print("=" * 60)
print("  STEP 8 — GBM LIFT TABLE (TEST SET)")
print("=" * 60)
print(gbm_lift_df.to_string(index=False))
print(f"\n  Peak KS (lift table) : {gbm_lift_df['ks_pct'].max():.2f}%  at decile {gbm_lift_df['ks_pct'].idxmax()+1}")

# ── Lift table as list of dicts for JSON export ───────────────
lift_records = gbm_lift_df.to_dict(orient="records")

  STEP 8 — GBM LIFT TABLE (TEST SET)
 decile  n  bads  goods  bad_rate_pct  lift  cum_bad_pct  cum_good_pct  ks_pct
      1  6     5      1         83.33  1.61        16.13          3.45   12.68
      2  6     4      2         66.67  1.29        29.03         10.34   18.69
      3  6     6      0        100.00  1.94        48.39         10.34   38.04
      4  6     2      4         33.33  0.65        54.84         24.14   30.70
      5  6     3      3         50.00  0.97        64.52         34.48   30.03
      6  6     2      4         33.33  0.65        70.97         48.28   22.69
      7  6     4      2         66.67  1.29        83.87         55.17   28.70
      8  6     3      3         50.00  0.97        93.55         65.52   28.03
      9  6     1      5         16.67  0.32        96.77         82.76   14.02
     10  6     1      5         16.67  0.32       100.00        100.00    0.00

  Peak KS (lift table) : 38.04%  at decile 3


In [31]:
feat_imp = pd.Series(gbm.feature_importances_, index=FEATURE_COLS)
feat_imp = feat_imp.sort_values(ascending=False)

print("=" * 60)
print("  STEP 9 — FEATURE IMPORTANCE (TOP 20)")
print("=" * 60)
print(feat_imp.head(20).round(4).to_string())

# ── Top 25 feature importance chart ──────────────────────────
top25 = feat_imp.head(25)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top25.index[::-1], top25.values[::-1], color="#1a6faf", edgecolor="white")
ax.set_xlabel("Feature Importance (MDI)", fontsize=10)
ax.set_title(f"{MODEL_NAME}\nTop 25 Feature Importances", fontsize=11, fontweight="bold")
ax.tick_params(axis="y", labelsize=8)
for bar in bars:
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.4f}", va="center", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "feature_importance_top25.png"), dpi=150)
plt.close()
print(f"\n  Saved → {MODELS_DIR}feature_importance_top25.png")

# ── Full importance chart ─────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, max(6, len(FEATURE_COLS)//4)))
feat_imp.sort_values().plot(kind="barh", ax=ax2, color="#1a6faf", edgecolor="white")
ax2.set_xlabel("Feature Importance (MDI)", fontsize=9)
ax2.set_title(f"{MODEL_NAME}\nAll Feature Importances", fontsize=10, fontweight="bold")
ax2.tick_params(axis="y", labelsize=6)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "feature_importance.png"), dpi=150)
plt.close()
print(f"  Saved → {MODELS_DIR}feature_importance.png")

  STEP 9 — FEATURE IMPORTANCE (TOP 20)
max_dpd_12m_mob               0.1454
max_dpd_ever                  0.1027
total_overdue_amount          0.0617
max_overdue_amount            0.0342
itr_to_declared_ratio         0.0314
max_dpd_6m_mob                0.0306
total_bounces                 0.0297
sanction_amount               0.0288
total_sanctioned_limit        0.0244
max_balance                   0.0205
bounce_rate                   0.0200
net_cashflow_ratio            0.0200
enquiry_per_tradeline         0.0194
income_deviation_pct          0.0171
declared_to_verified_ratio    0.0165
tenure_months                 0.0157
requested_amount              0.0157
bureau_score                  0.0152
emi_to_income_ratio           0.0142
foir                          0.0139

  Saved → D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2feature_importance_top25.png
  Saved → D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-

In [33]:
fpr_gbm, tpr_gbm, _ = roc_curve(y_test, gbm_proba)
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, lr_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"{MODEL_NAME} — ROC Curve & Score Distribution", fontsize=11, fontweight="bold")

# Left: ROC curves
ax1 = axes[0]
ax1.plot(fpr_gbm, tpr_gbm, color="#1a6faf", lw=2,
         label=f"GBM  (AUC={gbm_auc:.4f}, Gini={gbm_gini:.4f})")
ax1.plot(fpr_lr,  tpr_lr,  color="#e07b30", lw=1.5, linestyle="--",
         label=f"LR   (AUC={lr_auc:.4f}, Gini={lr_gini:.4f})")
ax1.plot([0,1],[0,1], "k--", lw=0.8, alpha=0.4)
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curve"); ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Right: GBM score distribution by class
ax2 = axes[1]
ax2.hist(gbm_proba[y_test==0], bins=20, alpha=0.65, color="#2ca02c",
         label="Good (0)", density=True, edgecolor="white")
ax2.hist(gbm_proba[y_test==1], bins=20, alpha=0.65, color="#d62728",
         label="Bad (1)",  density=True, edgecolor="white")
ax2.axvline(0.5, color="black", linestyle="--", lw=1.2, label="Threshold 0.5")
ax2.set_xlabel("Predicted Probability"); ax2.set_ylabel("Density")
ax2.set_title("GBM Score Distribution"); ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "roc_and_score_dist.png"), dpi=150)
plt.close()
print(f"  Saved → {MODELS_DIR}roc_and_score_dist.png")

  Saved → D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2roc_and_score_dist.png


In [35]:
print("=" * 60)
print("  STEP 12 — SAVE MODELS & ARTEFACTS")
print("=" * 60)

# ── pkl files ─────────────────────────────────────────────────
joblib.dump(gbm,    os.path.join(MODELS_DIR, "gbm_model.pkl"))
joblib.dump(lr,     os.path.join(MODELS_DIR, "logistic_model.pkl"))
joblib.dump(scaler, os.path.join(MODELS_DIR, "scaler.pkl"))

print(f"  gbm_model.pkl       → saved")
print(f"  logistic_model.pkl  → saved")
print(f"  scaler.pkl          → saved")

# ── feature_list.json ─────────────────────────────────────────
with open(os.path.join(MODELS_DIR, "feature_list.json"), "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"  feature_list.json   → {len(FEATURE_COLS)} features saved")

# ── metrics.json ──────────────────────────────────────────────
metrics = {
    "run":             datetime.now().isoformat(),
    "train_size":      int(X_train.shape[0]),
    "test_size":       int(X_test.shape[0]),
    "default_rate_pct": round(float(y.mean() * 100), 4),
    "features":        len(FEATURE_COLS),
    "gbm": {
        "auc":     round(float(gbm_auc),  4),
        "gini":    round(float(gbm_gini), 4),
        "ks":      gbm_ks,
        "cv_mean": round(float(cv_scores.mean()), 4),
        "cv_std":  round(float(cv_scores.std()),  4),
    },
    "logistic": {
        "auc":  round(float(lr_auc),  4),
        "gini": round(float(lr_gini), 4),
        "ks":   lr_ks,
    },
}
with open(os.path.join(MODELS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print(f"  metrics.json        → saved")

# ── model_metadata.json ───────────────────────────────────────
psi_status = "RETRAIN" if metrics["gbm"]["ks"] < 0.20 else "STABLE"
metadata = {
    "model_name":       MODEL_NAME,
    "run":              metrics["run"],
    "auc":              metrics["gbm"]["auc"],
    "gini":             metrics["gbm"]["gini"],
    "ks":               metrics["gbm"]["ks"],
    "features":         len(FEATURE_COLS),
    "train_size":       metrics["train_size"],
    "test_size":        metrics["test_size"],
    "default_rate_pct": metrics["default_rate_pct"],
    "ks_status":        ks_status(gbm_ks),
    "auc_status":       auc_status(gbm_auc),
    "psi_status":       psi_status,
}
with open(os.path.join(MODELS_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  model_metadata.json → saved")

# ── validation_report_GBM.json ────────────────────────────────
cm_list = confusion_matrix(y_test, (gbm_proba >= 0.5).astype(int)).tolist()
val_report = {
    "model":            "GradientBoosting",
    "auc":              metrics["gbm"]["auc"],
    "gini":             metrics["gbm"]["gini"],
    "ks":               metrics["gbm"]["ks"],
    "ks_status":        ks_status(gbm_ks),
    "auc_status":       auc_status(gbm_auc),
    "confusion_matrix": cm_list,
    "lift_table":       lift_records,
}
with open(os.path.join(MODELS_DIR, "validation_report_GBM.json"), "w") as f:
    json.dump(val_report, f, indent=2)
print(f"  validation_report_GBM.json → saved")
print("=" * 60)

  STEP 12 — SAVE MODELS & ARTEFACTS
  gbm_model.pkl       → saved
  logistic_model.pkl  → saved
  scaler.pkl          → saved
  feature_list.json   → 86 features saved
  metrics.json        → saved
  model_metadata.json → saved
  validation_report_GBM.json → saved


In [37]:
print("=" * 60)
print(f"  {MODEL_NAME}")
print("  TRAINING COMPLETE")
print("=" * 60)
print(f"  Train size     : {X_train.shape[0]:,}")
print(f"  Test size      : {X_test.shape[0]:,}")
print(f"  Features used  : {len(FEATURE_COLS)}")
print()
print(f"  ── GBM ────────────────────────────────")
print(f"  AUC            : {gbm_auc:.4f}  [{auc_status(gbm_auc)}]")
print(f"  Gini           : {gbm_gini:.4f}")
print(f"  KS             : {gbm_ks:.4f}   [{ks_status(gbm_ks)}]")
print(f"  CV mean AUC    : {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}")
print()
print(f"  ── Logistic (benchmark) ────────────────")
print(f"  AUC            : {lr_auc:.4f}")
print(f"  Gini           : {lr_gini:.4f}")
print(f"  KS             : {lr_ks:.4f}")
print()
print("  ── Saved artefacts ─────────────────────")
for f in [
    "gbm_model.pkl", "logistic_model.pkl", "scaler.pkl",
    "feature_list.json", "metrics.json", "model_metadata.json",
    "validation_report_GBM.json",
    "feature_importance.png", "feature_importance_top25.png",
    "roc_and_score_dist.png", "lift_chart.png",
]:
    print(f"  {MODELS_DIR}{f}")
print("=" * 60)

  CreditRiskEngine_GBM_v2
  TRAINING COMPLETE
  Train size     : 240
  Test size      : 60
  Features used  : 86

  ── GBM ────────────────────────────────
  AUC            : 0.7330  [PASS]
  Gini           : 0.4661
  KS             : 0.4127   [GOOD]
  CV mean AUC    : 0.7820  ±  0.0840

  ── Logistic (benchmark) ────────────────
  AUC            : 0.6830
  Gini           : 0.3660
  KS             : 0.3393

  ── Saved artefacts ─────────────────────
  D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2gbm_model.pkl
  D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2logistic_model.pkl
  D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2scaler.pkl
  D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/credit-risk-engine/models2feature_list.json
  D:/Data Analyst/Projects/Credit_Risk_Engine_Project/credit-risk-engine-v2/